# Shor's Integer Factoring Algorithm Workbook

What is this workbook? A workbook is a collection of problems, accompanied by solutions to them. The explanations focus on the logical steps required to solve a problem; they illustrate the concepts that need to be applied to come up with a solution to the problem, explaining the mathematical steps required.

This workbook describes the solutions to the problems offered in the "Shor's Integer Factoring Algorithm" kata. Since the problems involve code implementations of the solutions, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [ ]:
# Execute this cell to prepare the test infrastructure
from psiqdk.workbench import Qubits


## Problem 1. Find the function period classically

The classical solution to the problem is straightforward: start with $a^0 = 1$ and keep multiplying it by $a$ modulo $N$ until the result cycles back to $1$ again. The number of multiplications done before then is the period of the function.

In [ ]:
def find_period_classical(n: int, a: int) -> int:
    a_x = 1
    for p in range(1, n):
        a_x = (a_x * a) % n
        if a_x == 1:
            return p

## Problem 2. Multiply number by 2 modulo 15

Let's consider the transformation of the four-bit binary notation of $x$ (in little-endian encoding) when it is multiplied by $2$ modulo $15$ as described. (Remember that $16 \textrm{ mod } 15 = 1$ and that we keep $15$ unchanged to make the transformation reversible.)

| $x$ | Binary $x$ | $2x \textrm{ mod } 15$ | Binary $2x$ |
| :-: | :--------: | :--------------------: | :---------: |
| 0   | 0000 | 0   | 0000 |
| 1   | 1000 | 2   | 0100 |
| 2   | 0100 | 4   | 0010 |
| 3   | 1100 | 6   | 0110 |
| 4   | 0010 | 8   | 0001 |
| ... | ...  | ... | ...  |
| 7   | 1110 | 14  | 0111 |
| 8   | 0001 | 16 = 1 | 1000 |
| 9   | 1001 | 18 = 3 | 1100 |
| 10  | 0101 | 20 = 5 | 1010 |
| ... | ...  | ...    | ...  |
| 14  | 0111 | 28 = 13 | 1011 |
| 15  | 1111 | 15 | 1111 |

You can see that multiplying a number by $2$ is the same as doing a circular bit shift to the right: three least significant bits are shifted one position to the right, and the most significant bit wraps around to become the least significant bit of the result.

To implement this transformation as a circuit, you can use a sequence of three SWAPs: swap the last (most significant) bit `x[3]` with the bit before it `x[2]`, then swap the two middle bits `x[2]` and `x[1]`, and finally swap the two least significant bits `x[1]` and `x[0]`.

In [ ]:
def multiply_by_2_mod_15(x: Qubits) -> None:
    for ind in range(x.num_qubits - 1, 0, -1):
        x[ind].swap(x[ind - 1])

## Problem 3. Multiply number by 4 modulo 15

You can think of multiplication by $4$ as a sequence of two multiplications by $2$. In binary notation, this means that it's a circular bit shift by two positions, implemented as a pair of SWAPs: swap `x[3]` with `x[1]` and `x[2]` with `x[0]`.

In [ ]:
def multiply_by_4_mod_15(x: Qubits) -> None:
    for ind in range(x.num_qubits - 1, 1, -1):
        x[ind].swap(x[ind - 2])

## Problem 4. Multiply number by $2^k$ modulo 15

Let's represent the integer $k$ as its binary notation:

$$k = k_0 \cdot 2^0 + k_1 \cdot 2^1 + k_2 \cdot 2^2 + ...$$

Then we can express the result of multiplying $x$ by $2^k$ as follows:

$$x \cdot 2^k = x \cdot 2^{k_0 \cdot 2^0 + k_1 \cdot 2^1 + k_2 \cdot 2^2 + ...} = x \cdot 2^{k_0 \cdot 2^0} \cdot 2^{k_1 \cdot 2^1} \cdot 2^{k_2 \cdot 2^2} \cdot ...= ... \left( \left( \left(x \cdot 2^{k_0 \cdot 2^0}\right) \cdot 2^{k_1 \cdot 2^1}\right) \cdot 2^{k_2 \cdot 2^2}\right) \cdot ...$$

In other words, we can perform multiplication one bit of $k$ at a time:

1. If the least significant bit of $k$ $k_0$ is $1$, multiply $x$ by $2^{2^0} = 2^1 = 2$, otherwise skip this step.
2. If the second least significant bit of $k$ $k_1$ is $1$, multiply the current value in register $x$ by $2^{2^1} = 2^2 = 4$, otherwise skip this step.
3. If the third least significant bit of $k$ $k_2$ is $1$, multiply the current value in register $x$ by $2^{2^2} = 2^4 = 16$, otherwise skip this step, and so on, until we run out of bits of $k$.

The first two steps can be implemented using solutions to problems 2 and 3, respectively. The other steps were covered in the "Analyze" exercise, in which we've seen that they are just identity transformations, so don't need to have code written for them.

In [ ]:
def multiply_by_2k_mod_15(x: Qubits, k: int) -> None:
    if k & 1:
        multiply_by_2_mod_15(x)
    if k & 2:
        multiply_by_4_mod_15(x)

> Copyright (c) 2026 PsiQuantum